# TP2 — Problem Settings and Data Generation  
## Inverse Parametrized System of Parabolic PDEs a 3D column

This notebook defines the **problem settings** for **Test Problem 2 (TP2)**, which addresses an **inverse parametrized system of parabolic PDEs** on a complex **3D column geometry**.

The purpose of this notebook is to:
- define the physical problem and its parameters,
- generate **simulated IoT-like boundary measurements**,
- acquire and preprocess the 3D geometry,
- produce the mesh and visualization-ready files required by the subsequent stages.

This notebook represents the **first stage of the TP2 pipeline** and prepares all the assets used in the offline, inverse, and online stages.

In [ ]:
import sys
from pyprojroot import here

PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

In [ ]:
from paths import INV_PATH, TP2_PATH, MODEL_PATH, COLUMN_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP2_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + COLUMN_PATH

model_name = "column.blend"

In [ ]:
# Create needed folders
import os

lst_folders = ["figures", "files", "models", "column_parabolic_inverse", "trainPOD"]

for name in lst_folders:
    os.makedirs(os.path.join(ABS_PATH, name), exist_ok=True)

## Libraries and Dependencies

We start by importing all the libraries required for:
- geometry handling and mesh generation,
- data management,
- preparation of visualization files.

In [ ]:
import torch
import pandas as pd

from model_acquisition import Blend2Pina, Blend2Mesh, Msh2Xdmf

## Numerical Precision

Double precision is enforced

In [ ]:
torch.set_default_dtype(torch.float64)

## Parametrized Physical Problem

We consider a 3D domain $\Omega \subseteq \mathbb{R}^3$ representing a **column geometry**, with boundary $\Gamma = \partial \Omega$.  
The physical phenomenon is described by the following differenzial problem:
$$
\begin{equation}
    \begin{cases}
        u_t \left(x, y, z, t \right) - \Delta u \left( x, y, z, t \right) + F \left( t \right) & \Omega \times \left[ 0,1 \right] \\
        u_b = B \left( x, y, z, t \right) & \partial \Omega \times \left[0, 1 \right] \\
        u_0 \left( x, y, z \right) = I \left( x, y, z \right) & \Omega
    \end{cases}
    \tag{1}
\end{equation}
$$
where
\begin{equation}
    F \left( t \right) = \left(
        \begin{array}{c}
            \lambda e^{\lambda t} \\
            \lambda e^{\lambda t} - 2 \left( \alpha + \beta + 1 \right)
        \end{array}
    \right), \qquad \Omega \times \left[ 0,1 \right] ,
    \tag{2}
\end{equation}

\begin{equation}
    B \left( x, y, z, t \right) = \left(
        \begin{array}{c}
            e^{\lambda t} + \alpha x + \beta y + z \\
            e^{\lambda t} + \alpha x^2 + \beta y^2 + z^2 
        \end{array}
    \right), \qquad \partial \Omega \times \left[ 0,1 \right] ,
    \tag{3}
\end{equation}

\begin{equation}
    I \left( x, y, z \right) = \left(
        \begin{array}{c}
            1 + \alpha x + \beta y + z \\
            1 + \alpha x^2 + \beta y^2 + z^2
        \end{array}
    \right), \qquad \Omega .
    \tag{4}
\end{equation}

The scalar field u depends on a set of unknown parameters $\mu = \left( \lambda, \alpha, \beta \right)$.

The analytical solution is consistent with boundary conditions.

## Generation of Data for the Inverse Problem

In this stage, we generate **simulated measurements** that emulate IoT sensor data.

Sensors are assumed to be located on the boundary of the 3D domain.  
The corresponding values of the physical field are computed using the analytical solution and stored for later use in the inverse PINN stage.

In [ ]:
column = Blend2Pina(LOAD_MODEL + model_name)

## Boundary Sampling

In [ ]:
num_points = 1_000

surface = column.boundary(time_interval=[0, 1])
points = surface.sample(num_points)

## Parameter Selection and Simulated Measurements

A reference set of parameters $\mu = \left( \lambda, \alpha, \beta \right)$ is fixed to generate the synthetic data.

Using the analytical solution:
- boundary values are computed at the selected sensor locations,
- the resulting measurements are stored in tabular form,
- these data will later be used as observations in the inverse PINN problem.

In [ ]:
par_lambda = .1
par_alpha = .2
par_beta = .5

u1 = torch.exp(par_lambda*points.extract('t')) + par_alpha*points.extract('x') + par_beta*points.extract('y') + points.extract('z')
u2 = torch.exp(par_lambda*points.extract('t')) + par_alpha*(points.extract('x')**2) + par_beta*(points.extract('y')**2) + points.extract('z')**2

## Data Storage

The simulated measurements and sensor locations are saved in CSV format.

This ensures:
- reproducibility of the inverse problem,
- decoupling between data generation and inference,
- reuse of the same dataset across multiple experiments.

In [ ]:
total_info = torch.concat(
    [points.tensor, u1, u2],
    1
)

df = pd.DataFrame(
    data=total_info.numpy(),
    columns=["x", "y", "z", "t", "u1", "u2"]
)

df.to_csv("./files/data.csv", sep=";")

## Mesh Generation and Visualization Files

The 3D column geometry is discretized to generate a computational mesh suitable for numerical simulations.

In addition:
- visualization-ready files (.xdmf and associated data) are produced,
- these files enable inspection of solutions and errors in ParaView,
- the same mesh is reused consistently across all TP1 stages.

In [ ]:
column_msh = Blend2Mesh(LOAD_MODEL + model_name, "column")

In [ ]:
column_msh.create_single_meshes(len_msh=0.01)

In [ ]:
column_xdmf = Msh2Xdmf("column.msh", "column")
column_xdmf.to_xdmf(num_refine=3)
column_xdmf.to_xdmf()